# LLM 6일차 (2) — Pydantic `BaseModel` 로 LangGraph 상태 정의하기 : 유효성 검사 · `Field` · 언제 검사하나

교재 `AI Agent` **p.88~91 · 94~95**(Pydantic 라이브러리 · BaseModel vs TypedDict · Field 함수 · 실습)를 같은 그래프로 실행하며 확인한 노트입니다. 수업 필기는 TypedDict 실습까지라, 교재 예시 코드(p.91)를 바탕으로 만들었습니다.

```python
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    tool_output: str | None = Field(default=None)
```

> **출력 읽는 법** — 오늘 실습은 LLM 을 부르지 않고 **노드 함수가 정해 둔 메시지를 돌려주는 모의 실행**이라 **API 키 없이 전부 실제로 실행**했다. 이 노트에는 `# 📼` 셀이 없다.

## 0. 설치 · 실행 방법

`pip install langgraph pydantic` (pydantic 은 langchain 과 함께 이미 설치됨 · 정리 환경 `pydantic 2.14.3`). **API 키 없이** 맨 위부터 모두 실행.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph pydantic

In [2]:
import json
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import add_messages, StateGraph, START, END

## 1. `BaseModel` — 들어오는 값을 검사한다 (교재 p.88~89)

In [3]:
class User(BaseModel):
    id: int
    name: str
    is_active: bool = True                       # 기본값

print(User(id=1, name='김철수'))
print(User(id='2', name='이영희'), '← 글자 "2" 는 숫자로 바꿔 받아 준다 (lax 모드)')
for bad in [dict(id='삼', name='박민수'), dict(name='최지우')]:
    try:
        User(**bad)
    except ValidationError as e:
        print(f'{bad} → 오류 {e.error_count()}개 :', [(err['loc'], err['type']) for err in e.errors()])

id=1 name='김철수' is_active=True
id=2 name='이영희' is_active=True ← 글자 "2" 는 숫자로 바꿔 받아 준다 (lax 모드)
{'id': '삼', 'name': '박민수'} → 오류 1개 : [(('id',), 'int_parsing')]
{'name': '최지우'} → 오류 1개 : [(('id',), 'missing')]


- `TypedDict` 는 틀린 값을 그대로 담지만, `BaseModel` 은 만들 때 **바로 오류**를 낸다. 다만 '2' → 2 처럼 바꿀 수 있는 값은 바꿔서 받는다(엄격히 하려면 `model_config = {'strict': True}`)

## 2. `Field()` — 기본값 · 제약 조건 · 설명 (교재 p.90)

In [4]:
class SearchPlan(BaseModel):
    query: str = Field(description='검색할 문장', min_length=2, max_length=50)
    max_results: int = Field(default=3, ge=1, le=10, description='가져올 결과 수')
    retries: int = Field(default=0, ge=0, le=3)

print(SearchPlan(query='nvidia 매출'))
for bad in [dict(query='n'), dict(query='nvidia 매출', max_results=20)]:
    try:
        SearchPlan(**bad)
    except ValidationError as e:
        print(f'{bad} →', e.errors()[0]['msg'])
print(json.dumps(SearchPlan.model_json_schema()['properties'], ensure_ascii=False, indent=1)[:420])

query='nvidia 매출' max_results=3 retries=0
{'query': 'n'} → String should have at least 2 characters
{'query': 'nvidia 매출', 'max_results': 20} → Input should be less than or equal to 10
{
 "query": {
  "description": "검색할 문장",
  "maxLength": 50,
  "minLength": 2,
  "title": "Query",
  "type": "string"
 },
 "max_results": {
  "default": 3,
  "description": "가져올 결과 수",
  "maximum": 10,
  "minimum": 1,
  "title": "Max Results",
  "type": "integer"
 },
 "retries": {
  "default": 0,
  "maximum": 3,
  "minimum": 0,
  "title": "Retries",
  "type": "integer"
 }
}


> **📝 교재 p.90 보충** '설명(description)을 추가하면 LLM 에게 전달되는 시스템 프롬프트의 일부로 포함된다' → **이 모델을 LLM 에게 건넬 때만**이다. Day 047 의 `with_structured_output(Sentiment)` 처럼 모델을 '답 형식' 으로 주면 위 JSON 스키마(설명 포함)가 LLM 에게 간다. LangGraph **상태**에 적은 description 은 LLM 에게 자동으로 전달되지 않는다 — 그래프 안에서 쓰는 형식일 뿐이다

## 3. 상태를 `BaseModel` 로 (교재 p.91 · p.94)

In [5]:
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)   # 리듀서는 TypedDict 와 같다
    tool_output: str | None = Field(default=None)                                        # 기본값 → 빠진 키 문제 없음

CALL_ID = 'call_search_001'

def call_llm(state: AgentState) -> dict:
    print('---LLM 호출 노드---', type(state).__name__, '받음')
    question = state.messages[-1].content                     # 📝 BaseModel 은 state['messages'] 가 아니라 state.messages
    return {'messages': [AIMessage(content='', tool_calls=[{'name': 'tavily_search', 'args': {'query': question}, 'id': CALL_ID}])]}

def call_tool(state: AgentState) -> dict:
    print('---도구 노드---')
    call = state.messages[-1].tool_calls[0]
    result = f"'{call['args']['query']}' 검색 완료 (모의 결과)"
    return {'messages': [ToolMessage(content=result, tool_call_id=call['id'])], 'tool_output': result}

def summarize_result(state: AgentState) -> dict:
    print('---요약 노드--- tool_output =', repr(state.tool_output))
    return {'messages': [AIMessage(content=f"도구 실행 결과({state.tool_output})를 요약했습니다.")]}

def build(nodes):
    g = StateGraph(AgentState)
    for name, fn in nodes:
        g.add_node(name, fn)
    names = [n for n, _ in nodes]
    for a, b in zip([START] + names, names + [END]):
        g.add_edge(a, b)
    return g.compile()

agent = build([('call_llm_node', call_llm), ('call_tool_node', call_tool), ('summarize_node', summarize_result)])
final_state = agent.invoke({'messages': [HumanMessage(content='nvidia 2026년도 2분기 매출액은?')]})
print('\ninvoke 결과의 자료형 :', type(final_state).__name__, '| 키 :', list(final_state))
print('마지막 메시지 :', final_state['messages'][-1].content)

---LLM 호출 노드--- AgentState 받음
---도구 노드---
---요약 노드--- tool_output = "'nvidia 2026년도 2분기 매출액은?' 검색 완료 (모의 결과)"

invoke 결과의 자료형 : dict | 키 : ['messages', 'tool_output']
마지막 메시지 : 도구 실행 결과('nvidia 2026년도 2분기 매출액은?' 검색 완료 (모의 결과))를 요약했습니다.


In [6]:
# 📝 TypedDict 식으로 state['messages'] 를 쓰면
def dict_style(state: AgentState) -> dict:
    return {'tool_output': state['messages'][-1].content}
try:
    build([('dict_style', dict_style)]).invoke({'messages': [HumanMessage(content='안녕')]})
except TypeError as e:
    print('TypeError :', e)

# 기본값 덕분에 도구 노드를 건너뛰어도 KeyError 가 아니다
out = build([('summarize_node', summarize_result)]).invoke({'messages': [HumanMessage(content='안녕')]})
print('마지막 메시지 :', out['messages'][-1].content)

TypeError : 'AgentState' object is not subscriptable
---요약 노드--- tool_output = None
마지막 메시지 : 도구 실행 결과(None)를 요약했습니다.


- 노드가 받는 `state` 는 **`AgentState` 객체** → `state.messages` · `state.tool_output` 처럼 점(.)으로 읽는다
- 📝 교재 p.89 표의 '결과물 타입 : Pydantic BaseModel 자료형' → 노드가 **받는** 상태는 BaseModel 이지만, `invoke()` 가 **돌려주는** 최종 상태는 **dict** 다(위 출력). 그래서 결과는 `final_state['messages']` 로 읽는다
- TypedDict 노트 §8 의 `KeyError` 는 기본값(`default=None`) 덕분에 사라졌다

## 4. 언제 검사하나 — 입력 · 다음 노드 · 마지막 노드

In [7]:
def wrong_output(state: AgentState) -> dict:
    return {'tool_output': 123}                       # str | None 자리에 숫자

def next_node(state: AgentState) -> dict:
    return {}

# ① 처음 넣는 값이 틀리면 → 바로 오류
try:
    build([('next_node', next_node)]).invoke({'messages': [HumanMessage(content='안녕')], 'tool_output': 456})
except ValidationError as e:
    print('① 입력       :', e.errors()[0]['loc'], e.errors()[0]['msg'])

# ② 노드가 틀린 값을 돌려주면 → '다음 노드' 가 상태를 받을 때 오류
try:
    build([('wrong_output', wrong_output), ('next_node', next_node)]).invoke({'messages': [HumanMessage(content='안녕')]})
except ValidationError as e:
    print('② 다음 노드  :', e.errors()[0]['loc'], e.errors()[0]['msg'])

# ③ 틀린 값을 돌려준 노드가 마지막이면 → 검사 없이 그대로 결과에
out = build([('wrong_output', wrong_output)]).invoke({'messages': [HumanMessage(content='안녕')]})
print('③ 마지막 노드 :', repr(out['tool_output']), type(out['tool_output']).__name__, '← 검사되지 않았다')

① 입력       : ('tool_output',) Input should be a valid string
② 다음 노드  : ('tool_output',) Input should be a valid string
③ 마지막 노드 : 123 int ← 검사되지 않았다


- LangGraph 의 Pydantic 검사는 **노드에 상태를 넣을 때**(입력) 일어난다. 노드가 돌려준 dict 자체는 검사하지 않고, 그 값이 **다음 노드에 들어갈 때** 걸린다. 마지막 노드의 출력은 아무도 받지 않으니 그대로 나간다
- 📝 교재 p.89 '실행 중에 잘못된 값이 들어오면 **즉시** 에러' → '**다음 노드가 받을 때**' 에러. 마지막 노드의 결과는 직접 확인해야 한다

## 5. TypedDict vs BaseModel — 확인한 결과로 다시 정리 (교재 p.89 표)

| | `TypedDict` | `BaseModel` |
|---|---|---|
| 제공 | `typing` (파이썬 내장) | `pydantic` (외부 라이브러리, langchain 과 함께 설치) |
| 틀린 자료형 | 그대로 담음 (TypedDict 노트 §3) | 오류 — 입력 · 다음 노드에서 |
| 빠진 키 · 기본값 | 기본값 없음 → 읽을 때 `KeyError` | `Field(default=…)` 로 채움 |
| 노드에서 읽기 | `state['messages']` | `state.messages` |
| `invoke()` 결과 | dict | **dict** (BaseModel 아님) |
| 속도 | 빠름 | 검사하는 만큼 느림 |

## 정리

| 할 일 | 코드 |
|---|---|
| 상태 | `class AgentState(BaseModel): messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)` |
| 제약 | `Field(default=3, ge=1, le=10, min_length=2, description='…')` |
| 노드 | `state.messages` 로 읽고, 바꿀 항목만 dict 로 돌려줌 |
| 검사 시점 | 그래프 입력 · 다음 노드 입력 (마지막 노드 출력은 검사 안 함) |